In [33]:
# ================================
# Safe installs — do NOT let this touch torch/torchvision/torchaudio
# Kaggle's preinstalled torch already matches the P100's CUDA/compute capability
# ================================
!pip uninstall -y torchao
!pip install -q peft --no-deps
!pip install -q trl --no-deps
!pip install -q accelerate

import torch, transformers, datasets, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# Sanity check CUDA actually works before doing anything else
x = torch.randn(100).cuda()
y = torch.randn(100).cuda()
z = torch.matmul(x, y)
print("✓ CUDA ops work:", z.item())

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
torch: 2.10.0+cu128
transformers: 5.0.0
datasets: 5.0.0
peft: 0.19.1
CUDA available: True
GPU: Tesla T4
✓ CUDA ops work: 1.355029582977295


In [34]:
# ================================
# Imports
# ================================
import os
import time
import json
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, IA3Config, TaskType, get_peft_model
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"

# Sanity check inputs are attached
print(os.listdir(DATA_ROOT))
print(os.listdir(f"{DATA_ROOT}/hi"))# ================================
# Load chosen LRs from Phase 2b
# ================================
with open("/kaggle/input/notebooks/venkatkolluu/04-hyperparameter-search/chosen_lrs.json", "r") as f:
    CHOSEN_LRS = json.load(f)

print("Chosen learning rates:", CHOSEN_LRS)

['manifest.json', 'te', 'hi']
['train_392702.parquet', 'train_500.parquet', 'train_1000.parquet', 'test.parquet', 'train_50.parquet', 'valid.parquet', 'train_100.parquet']
Chosen learning rates: {'dora': 5e-05, 'ia3': 1e-05, 'lora': 5e-05}


In [35]:
# ================================
# Load chosen LRs from Phase 2b
# ================================
with open("/kaggle/input/notebooks/venkatkolluu/04-hyperparameter-search/chosen_lrs.json", "r") as f:
    CHOSEN_LRS = json.load(f)

print("Chosen learning rates:", CHOSEN_LRS)

Chosen learning rates: {'dora': 5e-05, 'ia3': 1e-05, 'lora': 5e-05}


In [36]:
# ================================
# Sweep configuration — full 90-run grid
# ================================
METHODS = ["lora", "dora", "ia3"]
LANGUAGES = ["hi", "te"]
BUDGETS = [50, 100, 500, 1000, 392702]
SEEDS = [42, 123, 456]
EPOCHS = 5
BATCH_SIZE = 16

In [37]:
# ================================
# Data loading
# ================================
def build_loaders(language, budget, seed):
    train_df = pd.read_parquet(f"{DATA_ROOT}/{language}/train_{budget}.parquet")
    valid_df = pd.read_parquet(f"{DATA_ROOT}/{language}/valid.parquet")

    def tokenize(batch):
        return tokenizer(
            batch["premise"], batch["hypothesis"],
            truncation=True, padding="max_length", max_length=MAX_LENGTH
        )

    train_ds = Dataset.from_pandas(train_df).rename_column("label", "labels")
    valid_ds = Dataset.from_pandas(valid_df).rename_column("label", "labels")

    train_ds = train_ds.map(tokenize, batched=True)
    valid_ds = valid_ds.map(tokenize, batched=True)

    keep = ["input_ids", "attention_mask", "labels"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep])

    train_ds.set_format("torch")
    valid_ds.set_format("torch")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=32)

    return train_loader, valid_loader

In [38]:
# ================================
# Model builders
# ================================
def load_base_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    ).cuda()

def build_model(method):
    base = load_base_model()

    if method == "lora":
        config = LoraConfig(
            r=8, lora_alpha=16, lora_dropout=0.1, bias="none",
            task_type=TaskType.SEQ_CLS, target_modules=["query", "value"]
        )
        return get_peft_model(base, config)

    if method == "dora":
        config = LoraConfig(
            r=8, lora_alpha=16, lora_dropout=0.1, bias="none", use_dora=True,
            task_type=TaskType.SEQ_CLS, target_modules=["query", "value"]
        )
        return get_peft_model(base, config)

    if method == "ia3":
        config = IA3Config(
            task_type=TaskType.SEQ_CLS,
            target_modules=["key", "value", "output.dense"],
            feedforward_modules=["output.dense"]
        )
        return get_peft_model(base, config)

    raise ValueError(f"Unknown method: {method}")

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [39]:
# ================================
# Train + evaluate one configuration
# ================================
def train_and_evaluate(method, language, budget, seed, epochs=EPOCHS):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

    train_loader, valid_loader = build_loaders(language, budget, seed)

    model = build_model(method)
    lr = CHOSEN_LRS[method]
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )

    trainable_params = count_trainable_params(model)

    model.train()
    start = time.perf_counter()

    for epoch in range(epochs):
        for batch in train_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    train_time = time.perf_counter() - start

    torch.cuda.reset_peak_memory_stats()
    model.train()
    with torch.no_grad():
        batch = next(iter(train_loader))
        batch = {k: v.cuda() for k, v in batch.items()}
        _ = model(**batch)
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3

    model.eval()
    predictions, labels = [], []
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")

    del model
    torch.cuda.empty_cache()

    return {
        "method": method,
        "language": language,
        "budget": budget,
        "seed": seed,
        "lr": lr,
        "epochs": epochs,
        "accuracy": round(accuracy, 6),
        "macro_f1": round(macro_f1, 6),
        "trainable_params": trainable_params,
        "peak_gpu_memory_gb": round(peak_memory_gb, 4),
        "training_time_sec": round(train_time, 2),
    }

In [ ]:
# ================================
# Main loop — run all 90 configurations
# ================================
results_file = "/kaggle/working/experiment_results.jsonl"
results = []

total_runs = len(METHODS) * len(LANGUAGES) * len(BUDGETS) * len(SEEDS)
current_run = 0

for method in METHODS:
    for language in LANGUAGES:
        for budget in BUDGETS:
            for seed in SEEDS:
                current_run += 1
                print(f"\n[{current_run}/{total_runs}] {method} | {language} | budget={budget} | seed={seed}")

                try:
                    result = train_and_evaluate(method, language, budget, seed, epochs=EPOCHS)
                    results.append(result)
                    print(f"  ✓ Acc: {result['accuracy']:.4f}  F1: {result['macro_f1']:.4f}  Time: {result['training_time_sec']:.1f}s")

                    with open(results_file, "a") as f:
                        f.write(json.dumps(result) + "\n")

                except Exception as e:
                    print(f"  ✗ ERROR: {e}")
                    with open(results_file, "a") as f:
                        f.write(json.dumps({
                            "method": method,
                            "language": language,
                            "budget": budget,
                            "seed": seed,
                            "error": str(e)
                        }) + "\n")

print(f"\n✓ Sweep finished — {total_runs} configurations attempted.")


[1/90] lora | hi | budget=50 | seed=42


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

  ✓ Acc: 0.3333  F1: 0.1667  Time: 70.5s

[13/90] lora | hi | budget=392702 | seed=42


Map:   0%|          | 0/392702 [00:00<?, ? examples/s]

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
